# Bull Flag Trailing Exit Backtest

这份 notebook 直接读取 `Dataframes/stock_price.csv`，研究当前 bull flag 主推版本：

- bull flag entry
- `TP1 -> trailing stop` 动态退出

默认参数使用当前研究里最平衡的一版：

- `max_flag_retrace_ratio = 0.30`
- `min_breakout_body_pct = 0.60`
- `max_breakout_upper_shadow_pct = 0.35`
- `max_breakout_lower_shadow_pct = 0.50`
- `max_peak_sma60_return_10 = 0.055`
- `tp1_fraction_of_target = 0.50`
- `trailing_stop_fraction_of_flagpole = 0.25`

这份 notebook 会做 4 件事：

1. 生成动态退出版本的 `trade_df`
2. 跑回测并展示核心指标
3. 和静态 bull flag 基线做一张对照表
4. 抽一笔样本交易，查看 `exit_path` 和图形上下文


In [ ]:
# Notebook bootstrap: repo root + sys.path
import sys
from pathlib import Path

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (path for path in candidate_roots if (path / "strategies").exists() and (path / "backtester").exists() and (path / "Dataframes").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate repo root from the current notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from backtester import TradePlanBacktester
from strategies.bull_flag_continuation import BullFlagContinuationResearcher, BullFlagStrategyConfig
from strategies.bull_flag_exit_variants import BullFlagDynamicExitConfig, BullFlagTrailingAfterTp1Researcher

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((path for path in candidate_roots if (path / 'Dataframes' / 'stock_price.csv').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate repo root from the current notebook working directory.')

PRICE_PATH = REPO_ROOT / 'Dataframes' / 'stock_price.csv'
REPO_ROOT, PRICE_PATH

In [ ]:
stock_candle_df = pd.read_csv(PRICE_PATH)
if 'Unnamed: 0' in stock_candle_df.columns:
    stock_candle_df = stock_candle_df.drop(columns=['Unnamed: 0'])

stock_candle_df['date'] = pd.to_datetime(stock_candle_df['date'])
if 'constituent_trade_date' in stock_candle_df.columns:
    stock_candle_df['constituent_trade_date'] = pd.to_datetime(stock_candle_df['constituent_trade_date'])

stock_candle_df = stock_candle_df.sort_values(['ticker', 'date']).reset_index(drop=True)
stock_candle_df.head()

In [ ]:
dynamic_config = BullFlagDynamicExitConfig(
    max_flag_retrace_ratio=0.30,
    min_breakout_body_pct=0.60,
    max_breakout_upper_shadow_pct=0.35,
    max_breakout_lower_shadow_pct=0.50,
    max_peak_sma60_return_10=0.055,
    tp1_fraction_of_target=0.50,
    trailing_stop_fraction_of_flagpole=0.25,
)

researcher = BullFlagTrailingAfterTp1Researcher(stock_candle_df, config=dynamic_config)
trade_df = researcher.add_trade_df().copy()

pd.Series({
    'rows': len(researcher.stock_candle_df),
    'planned_trade_count': len(trade_df),
    'date_min': researcher.stock_candle_df['date'].min(),
    'date_max': researcher.stock_candle_df['date'].max(),
    'exit_reason_counts': trade_df['exit_reason'].value_counts(dropna=False).to_dict(),
})

## 回测

这里直接把 `trade_df` 传给 `TradePlanBacktester`，避免重复重算 researcher。

In [ ]:
backtester = TradePlanBacktester(
    stock_candle_df,
    trade_df=trade_df,
    initial_capital=1_000_000.0,
    fixed_entry_notional=20_000.0,
    board_lot_size=100,
)

results = backtester.compute_metrics(start_date='2020-01-01', end_date='2026-03-16')
summary = pd.Series(results['summary'])
trade_summary = pd.Series(results.get('trade_summary', {}))

display(summary)
display(trade_summary)
results['figure']

## 和静态 bull flag 基线对照

同一套 entry 环境下，对比静态止盈和默认 trailing 退出版。

In [ ]:
static_config = BullFlagStrategyConfig(
    max_flag_retrace_ratio=0.30,
    min_breakout_body_pct=0.60,
    max_breakout_upper_shadow_pct=0.35,
    max_breakout_lower_shadow_pct=0.50,
    max_peak_sma60_return_10=0.055,
)
static_researcher = BullFlagContinuationResearcher(stock_candle_df, config=static_config)
static_trade_df = static_researcher.add_trade_df().copy()

static_backtester = TradePlanBacktester(
    stock_candle_df,
    trade_df=static_trade_df,
    initial_capital=1_000_000.0,
    fixed_entry_notional=20_000.0,
    board_lot_size=100,
)
static_results = static_backtester.compute_metrics(start_date='2020-01-01', end_date='2026-03-16')

compare_metrics = [
    'planned_trade_count',
    'trade_win_rate',
    'average_trade_return',
    'profit_factor',
    'total_return',
    'sharpe',
    'max_drawdown',
]
comparison_df = pd.DataFrame([
    {'variant': 'baseline_static', **{metric: static_results['summary'].get(metric) for metric in compare_metrics}},
    {'variant': 'tp1_trailing_default', **{metric: results['summary'].get(metric) for metric in compare_metrics}},
])
comparison_df

## 动态退出特征概览

看一下这版动态退出到底把交易分成了什么样。

In [ ]:
dynamic_columns = [
    'ticker',
    'signal_date',
    'entry_date',
    'exit_date',
    'exit_reason',
    'tp1_price',
    'tp1_reached',
    'tp1_hit_date',
    'post_tp1_stop_price',
    'pnl_pct',
]

display(trade_df[dynamic_columns].head(10))
display(trade_df['exit_reason'].value_counts(dropna=False).rename('count').to_frame())
display(pd.Series({
    'tp1_reached_ratio': float(trade_df['tp1_reached'].fillna(False).mean()) if not trade_df.empty else float('nan'),
    'average_tp1_price': float(pd.to_numeric(trade_df['tp1_price'], errors='coerce').mean()) if not trade_df.empty else float('nan'),
}))

## 单笔样本复盘

优先挑一笔 `trailing_stop` 退出的交易；如果没有，就退化到第一笔。

In [ ]:
if trade_df.empty:
    sample_signal = None
else:
    trailing_candidates = trade_df[trade_df['exit_reason'].astype(str).eq('trailing_stop')]
    sample_signal = trailing_candidates.iloc[0] if not trailing_candidates.empty else trade_df.iloc[0]

sample_signal

In [ ]:
if sample_signal is not None:
    inspection = researcher.inspect_signal(sample_signal['ticker'], sample_signal['signal_date'])
    display(pd.Series(inspection['summary']))
    display(inspection['signal_row'])
    display(inspection['exit_path'])
    researcher.plot_signal_context(sample_signal['ticker'], sample_signal['signal_date'])
else:
    print('No trade found in trade_df.')

## 前 N 个最赚钱 / 最亏钱交易

这里把已平仓交易里最赚钱和最亏钱的前 N 笔都拉出来，逐笔展示：

- 交易摘要
- `inspect_signal()` 的 summary
- `exit_path`
- `plot_signal_context()` 图形

这样可以比较直观地看出：

- 最好的交易通常长什么样
- 最差的交易通常失败在哪
- trailing exit 是怎么保护利润、或者没保护住的


In [ ]:
TOP_N = 5
EXTREME_LOOKBACK = 80
EXTREME_LOOKAHEAD = 20

closed_trades = trade_df.copy()
if 'trade_status' in closed_trades.columns:
    closed_trades = closed_trades[closed_trades['trade_status'].astype(str).eq('closed')].copy()

closed_trades['pnl_pct_num'] = pd.to_numeric(closed_trades['pnl_pct'], errors='coerce')
closed_trades = closed_trades[closed_trades['pnl_pct_num'].notna()].copy()

summary_columns = [
    'ticker',
    'signal_date',
    'entry_date',
    'exit_date',
    'exit_reason',
    'pnl_pct',
    'reward_to_risk',
    'flag_retrace_ratio',
    'flagpole_return',
    'holding_days',
    'tp1_reached',
]
summary_columns = [column for column in summary_columns if column in closed_trades.columns]

best_trades = closed_trades.nlargest(TOP_N, 'pnl_pct_num').reset_index(drop=True)
worst_trades = closed_trades.nsmallest(TOP_N, 'pnl_pct_num').reset_index(drop=True)

display(best_trades[summary_columns])
display(worst_trades[summary_columns])

In [ ]:
def review_trade_block(title, trades):
    print(title)
    if trades.empty:
        print('No closed trades available.')
        return

    for rank, (_, trade) in enumerate(trades.iterrows(), start=1):
        print(
            f"[{rank}] {trade['ticker']} | signal={trade['signal_date']} | "
            f"exit={trade.get('exit_reason', None)} | pnl_pct={trade.get('pnl_pct', None)}"
        )
        display(trade[summary_columns].to_frame().T)
        inspection = researcher.inspect_signal(
            trade['ticker'],
            trade['signal_date'],
            lookback=EXTREME_LOOKBACK,
            lookahead=EXTREME_LOOKAHEAD,
        )
        display(pd.Series(inspection['summary']))
        display(inspection['exit_path'])
        display(
            researcher.plot_signal_context(
                trade['ticker'],
                trade['signal_date'],
                lookback=EXTREME_LOOKBACK,
                lookahead=EXTREME_LOOKAHEAD,
            )
        )

review_trade_block('Top winners', best_trades)
review_trade_block('Top losers', worst_trades)